# ECRS — Wave 0: Data Engineering & Simulation Setup (v2)

**Employer Contribution Risk Score (ECRS)** — BPJS Kesehatan Healthkathon 2026, kategori *Efisiensi Risiko pada Pemberi Kerja*.

Notebook ini bikin **dataset sintetis skala ~1.000 badan usaha** untuk 3 modul deteksi (Module A/B/C), lengkap dengan `ground_truth.csv` (kasus anomali yang sengaja disisipkan) supaya bisa dipakai validasi kuantitatif (precision/recall/false-positive rate) di Wave 1.

**Revisi dari v1** (setelah review internal):
1. Skala naik dari 27 -> **~1.020 employer**, cohort minimal **34 anggota** (dulu cuma 3) -- supaya peer-group benchmarking (Module B) valid secara statistik, bukan cohort of one/two/three.
2. Tambah tabel **`resign_records`** -- dulu tidak ada sama sekali, padahal rule "suppress kalau ada resign record matching" di Module A butuh data ini untuk bisa diuji.
3. Tambah kasus **true-negative** (`CLEAN_MUTASI_SAH`): headcount turun tajam TAPI resign record-nya cocok -- Module A harus TIDAK men-flag ini.
4. Jumlah kasus anomali per kategori naik ke **25** (dari cuma 2) -- cukup untuk laporan precision/recall yang berarti, bukan "kebetulan ketangkep 1 kasus".

**Cara pakai di Google Colab:** Runtime → Run all. Semua library (`pandas`, `numpy`) sudah tersedia default di Colab. Output CSV kesimpan di folder `ecrs_wave0_data/`, sel paling bawah otomatis nawarin download `.zip` kalau dijalankan di Colab.

**Catatan jujur:** dataset ini 100% sintetis/dummy dengan angka yang plausible — bukan data BPJS asli. Tujuannya: kasih Module A/B/C (Wave 1) sesuatu yang nyata buat dihitung dan divalidasi, bukan angka hardcoded seperti di prototype UI.

## 0. Config & reproducibility

Seed di-fix supaya hasil generate konsisten tiap kali notebook dijalankan ulang.

In [1]:
import numpy as np
import pandas as pd
import random
import os
import time

RNG_SEED = 42
np.random.seed(RNG_SEED)
random.seed(RNG_SEED)

OUT_DIR = "ecrs_wave0_data"
os.makedirs(OUT_DIR, exist_ok=True)

PERIODS = [f"2025-{m:02d}" for m in range(1, 13)]  # 12 periode bulanan
CONTRIBUTION_RATE = 0.04  # simplifikasi: expected_contribution = DPI * headcount * rate

print(f"Jumlah periode: {len(PERIODS)} ({PERIODS[0]} s.d. {PERIODS[-1]})")

Jumlah periode: 12 (2025-01 s.d. 2025-12)


## 1. Definisi cohort, sektor, wilayah, skala

Cohort untuk Module B = kombinasi **(sektor_usaha, wilayah, skala)**. Dibangun dari full cartesian **6 sektor x 5 wilayah = 30 kombinasi tetap**, tiap kombinasi diberi satu skala dominan secara siklik (bukan random) supaya distribusi skala antar-cohort tetap merata.

Dengan **34 employer per cohort**, tiap peer-group benchmarking dibandingkan terhadap ~33 peer (bukan 2 seperti versi awal) — ini yang bikin median/IQR cohort benar-benar bermakna secara statistik.

In [2]:
SEKTOR_LIST = ["Manufaktur", "Konstruksi", "Perdagangan & Ritel",
               "Jasa Keuangan", "Teknologi & Digital", "Perkebunan & Agribisnis"]
WILAYAH_LIST = ["DKI Jakarta", "Jawa Barat", "Jawa Timur", "Sumatera Utara", "Sulawesi Selatan"]
SKALA_LIST = ["Kecil", "Menengah", "Besar"]

COHORTS = []
for si, sektor in enumerate(SEKTOR_LIST):
    for wi, wilayah in enumerate(WILAYAH_LIST):
        skala = SKALA_LIST[(si * len(WILAYAH_LIST) + wi) % len(SKALA_LIST)]
        COHORTS.append((sektor, wilayah, skala))

EMPLOYERS_PER_COHORT = 34  # 30 cohort x 34 = 1020 employer
N_TOTAL = len(COHORTS) * EMPLOYERS_PER_COHORT

SKALA_HEADCOUNT_RANGE = {"Kecil": (15, 60), "Menengah": (60, 250), "Besar": (250, 900)}

# Basis upah per sektor (angka plausible untuk simulasi, BUKAN data resmi BPJS/pemerintah)
BASE_WAGE_BY_SEKTOR = {
    "Manufaktur": 4_800_000, "Konstruksi": 4_500_000, "Perdagangan & Ritel": 3_800_000,
    "Jasa Keuangan": 7_500_000, "Teknologi & Digital": 8_500_000, "Perkebunan & Agribisnis": 3_400_000,
}
WILAYAH_WAGE_INDEX = {
    "DKI Jakarta": 1.15, "Jawa Barat": 1.00, "Jawa Timur": 0.95,
    "Sumatera Utara": 0.90, "Sulawesi Selatan": 0.88,
}
SKALA_WAGE_MULTIPLIER = {"Kecil": 0.90, "Menengah": 1.00, "Besar": 1.12}

# Jumlah kasus per kategori anomali
N_COLD_START = 15
N_PDUK_FRAUD = 25          # drop tajam TANPA resign record matching -> harus diflag
N_PDUK_TRUE_NEGATIVE = 10  # drop tajam DENGAN resign record matching -> TIDAK boleh diflag
N_WAGE_UNDERREPORT = 25
N_REMIT_GAP = 25

print(f"Target total employer: {N_TOTAL} ({len(COHORTS)} cohort x {EMPLOYERS_PER_COHORT})")

Target total employer: 1020 (30 cohort x 34)


## 2. `employer_master` + pembagian kelompok kasus

Employer diacak lalu dibagi ke kelompok: cold-start, PDUK fraud, PDUK legit (true-negative), under-reporting wage, remittance gap, dan sisanya clean/counter-example — semuanya **saling eksklusif** supaya ground truth tidak ambigu waktu dipakai hitung precision/recall per modul.

In [3]:
def gen_employer_master():
    rows = []
    idx = 1
    for sektor, wilayah, skala in COHORTS:
        for _ in range(EMPLOYERS_PER_COHORT):
            rows.append({
                "employer_id": f"EMP-{idx:04d}",
                "sektor_usaha": sektor,
                "wilayah": wilayah,
                "skala": skala,
                "tanggal_registrasi": "2022-01-01",
            })
            idx += 1
    return pd.DataFrame(rows)


employer_master = gen_employer_master()
all_ids = employer_master["employer_id"].tolist()
random.shuffle(all_ids)

pool = list(all_ids)
cold_start_ids = pool[:N_COLD_START]; pool = pool[N_COLD_START:]
pduk_fraud_ids = pool[:N_PDUK_FRAUD]; pool = pool[N_PDUK_FRAUD:]
pduk_legit_ids = pool[:N_PDUK_TRUE_NEGATIVE]; pool = pool[N_PDUK_TRUE_NEGATIVE:]
wage_underreport_ids = pool[:N_WAGE_UNDERREPORT]; pool = pool[N_WAGE_UNDERREPORT:]
remit_gap_ids = pool[:N_REMIT_GAP]; pool = pool[N_REMIT_GAP:]
clean_ids = pool

employer_master.loc[employer_master["employer_id"].isin(cold_start_ids), "tanggal_registrasi"] = "2025-11-01"

def periods_for_employer(employer_id):
    if employer_id in cold_start_ids:
        return PERIODS[-2:]
    return PERIODS

print(f"cold_start={len(cold_start_ids)}  pduk_fraud={len(pduk_fraud_ids)}  pduk_legit={len(pduk_legit_ids)}  "
      f"wage_underreport={len(wage_underreport_ids)}  remit_gap={len(remit_gap_ids)}  clean={len(clean_ids)}")
employer_master.head()

cold_start=15  pduk_fraud=25  pduk_legit=10  wage_underreport=25  remit_gap=25  clean=920


,employer_id,sektor_usaha,wilayah,skala,tanggal_registrasi
0,EMP-0001,Manufaktur,DKI Jakarta,Kecil,2022-01-01
1,EMP-0002,Manufaktur,DKI Jakarta,Kecil,2022-01-01
2,EMP-0003,Manufaktur,DKI Jakarta,Kecil,2022-01-01
3,EMP-0004,Manufaktur,DKI Jakarta,Kecil,2022-01-01
4,EMP-0005,Manufaktur,DKI Jakarta,Kecil,2022-01-01


## 3. `headcount_timeseries`

Random-walk kecil (~2%/bulan) di sekitar baseline sesuai skala perusahaan.

In [4]:
def gen_headcount(employer_master):
    rows = []
    for _, emp in employer_master.iterrows():
        lo, hi = SKALA_HEADCOUNT_RANGE[emp["skala"]]
        hc = random.randint(lo, hi)
        for p in periods_for_employer(emp["employer_id"]):
            hc = max(1, round(hc * (1 + np.random.normal(0, 0.02))))
            rows.append({"employer_id": emp["employer_id"], "periode": p, "jumlah_peserta_aktif": hc})
    return pd.DataFrame(rows)


headcount_df = gen_headcount(employer_master)
headcount_df.head()

,employer_id,periode,jumlah_peserta_aktif
0,EMP-0001,2025-01,49
1,EMP-0001,2025-02,49
2,EMP-0001,2025-03,50
3,EMP-0001,2025-04,52
4,EMP-0001,2025-05,52


## 4. `payroll_timeseries` (DPI)

Upah dasar = f(sektor, wilayah, skala) + faktor unik per-employer (±5%) + noise bulanan (~1.5%). Employer dalam cohort yang sama sengaja dibuat mirip upahnya — prasyarat supaya peer-group benchmarking masuk akal.

In [5]:
def base_wage(sektor, wilayah, skala):
    return BASE_WAGE_BY_SEKTOR[sektor] * WILAYAH_WAGE_INDEX[wilayah] * SKALA_WAGE_MULTIPLIER[skala]


def gen_payroll(employer_master):
    rows = []
    for _, emp in employer_master.iterrows():
        base = base_wage(emp["sektor_usaha"], emp["wilayah"], emp["skala"]) * np.random.uniform(0.95, 1.05)
        for p in periods_for_employer(emp["employer_id"]):
            dpi = base * (1 + np.random.normal(0, 0.015))
            rows.append({"employer_id": emp["employer_id"], "periode": p, "rata2_DPI": round(dpi, -2)})
    return pd.DataFrame(rows)


payroll_df = gen_payroll(employer_master)
payroll_df.head()

,employer_id,periode,rata2_DPI
0,EMP-0001,2025-01,4993700.0
1,EMP-0001,2025-02,4820000.0
2,EMP-0001,2025-03,4817900.0
3,EMP-0001,2025-04,4884900.0
4,EMP-0001,2025-05,4870600.0


## 5. `remittance_timeseries` (expected vs actual)

`expected_contribution = DPI x headcount x tarif` (tarif disederhanakan 4% untuk simulasi). `actual_remittance` dibuat mendekati expected dengan noise kecil.

In [6]:
def gen_remittance(payroll_df, headcount_df):
    merged = payroll_df.merge(headcount_df, on=["employer_id", "periode"])
    merged["expected_contribution"] = (merged["rata2_DPI"] * merged["jumlah_peserta_aktif"] * CONTRIBUTION_RATE).round(-3)
    noise = np.random.normal(0, 0.01, size=len(merged))
    merged["actual_remittance"] = (merged["expected_contribution"] * (1 + noise)).round(-3)
    return merged[["employer_id", "periode", "expected_contribution", "actual_remittance"]]


remittance_df = gen_remittance(payroll_df, headcount_df)
remittance_df.head()

,employer_id,periode,expected_contribution,actual_remittance
0,EMP-0001,2025-01,9788000.0,9784000.0
1,EMP-0001,2025-02,9447000.0,9398000.0
2,EMP-0001,2025-03,9636000.0,9484000.0
3,EMP-0001,2025-04,10161000.0,10047000.0
4,EMP-0001,2025-05,10131000.0,10204000.0


## 6. `resign_records` (BARU di v2)

**Ini yang sebelumnya hilang total dari Wave 0** — padahal Module A butuh data ini untuk rule "suppress flag kalau ada resign record resmi yang match dengan penurunan headcount". Tanpa tabel ini, aturan itu cuma tertulis di dokumen desain tapi nggak pernah bisa diuji.

Baseline: turnover wajar kecil tiap periode (~1% headcount, independen dari fluktuasi headcount lain). Untuk kasus PDUK (fraud dan legit) di bawah, angka ini akan di-override secara spesifik.

In [7]:
def gen_resign_records(headcount_df):
    rows = []
    for _, r in headcount_df.iterrows():
        lam = max(0.3, r["jumlah_peserta_aktif"] * 0.01)
        keluar = int(np.random.poisson(lam))
        rows.append({"employer_id": r["employer_id"], "periode": r["periode"], "jumlah_keluar": keluar})
    return pd.DataFrame(rows)


resign_df = gen_resign_records(headcount_df)
resign_df.head()

,employer_id,periode,jumlah_keluar
0,EMP-0001,2025-01,0
1,EMP-0001,2025-02,0
2,EMP-0001,2025-03,1
3,EMP-0001,2025-04,1
4,EMP-0001,2025-05,0


## 7. Injeksi labeled anomaly cases + `ground_truth`

Empat jenis kasus disisipkan (**saling eksklusif**, masing-masing kelompok tidak overlap):

- **PDUK fraud** (25 kasus) — headcount drop tajam (30-55%), resign record di periode itu SENGAJA dibuat 0-1 → tidak ada catatan resmi yang menjelaskan hilangnya peserta → harus diflag Module A.
- **PDUK legit / true-negative** (10 kasus) — headcount drop tajam yang sama, TAPI resign record-nya dibuat cocok dengan magnitude drop → Module A **harus tidak** men-flag ini (uji dua arah untuk confounder-handling, bukan cuma uji "berhasil deteksi").
- **Under-reporting wage** (25 kasus) — DPI ditekan ke 60-72% dari **median** upah peer di cohort yang sama (peer = anggota cohort selain diri sendiri).
- **Remittance gap** (25 kasus) — actual remittance 15-30% di bawah expected, berulang ≥2 periode berturut-turut.
- **Cold-start** (15 kasus) — baru registrasi, cuma 2 periode data.
- Sisanya (~920 employer) dibiarkan **bersih** sebagai counter-example untuk mengukur false-positive rate.

In [8]:
ground_truth = {e: {"anomaly_type": "CLEAN", "detail": "-"} for e in employer_master["employer_id"]}


def mark(emp_id, anomaly_type, detail):
    prev = ground_truth[emp_id]
    if prev["anomaly_type"] == "CLEAN":
        ground_truth[emp_id] = {"anomaly_type": anomaly_type, "detail": detail}
    else:
        ground_truth[emp_id] = {"anomaly_type": prev["anomaly_type"] + "+" + anomaly_type,
                                 "detail": prev["detail"] + " | " + detail}


for emp_id in cold_start_ids:
    mark(emp_id, "COLD_START", "Registrasi baru (2025-11), hanya 2 periode histori -> harus INSUFFICIENT_DATA di Module A")

print("Ground truth diinisialisasi untuk", len(ground_truth), "employer")

Ground truth diinisialisasi untuk 1020 employer


In [9]:
# --- 7a. PDUK fraud: drop tajam, TANPA resign record matching ---
for emp_id in pduk_fraud_ids:
    hc_mask = headcount_df["employer_id"] == emp_id
    sub = headcount_df.loc[hc_mask].sort_values("periode")
    break_idx = random.randint(5, 8)
    drop_pct = random.uniform(0.30, 0.55)
    idxs = sub.index.tolist()
    new_vals = sub["jumlah_peserta_aktif"].tolist()
    reduced_base = round(new_vals[break_idx] * (1 - drop_pct))
    for i in range(break_idx, len(new_vals)):
        new_vals[i] = max(1, round(reduced_base * (1 + np.random.normal(0, 0.02))))
    headcount_df.loc[idxs, "jumlah_peserta_aktif"] = new_vals

    r_mask = (resign_df["employer_id"] == emp_id) & (resign_df["periode"] == sub["periode"].iloc[break_idx])
    resign_df.loc[r_mask, "jumlah_keluar"] = random.randint(0, 1)

    mark(emp_id, "PDUK",
         f"Headcount drop ~{drop_pct:.0%} mulai periode {sub['periode'].iloc[break_idx]}, TANPA resign record matching")

print("PDUK fraud diinjeksi ke", len(pduk_fraud_ids), "employer")

PDUK fraud diinjeksi ke 25 employer


In [10]:
# --- 7b. PDUK legit (true negative): drop tajam, DENGAN resign record matching ---
for emp_id in pduk_legit_ids:
    hc_mask = headcount_df["employer_id"] == emp_id
    sub = headcount_df.loc[hc_mask].sort_values("periode")
    break_idx = random.randint(5, 8)
    drop_pct = random.uniform(0.30, 0.55)
    idxs = sub.index.tolist()
    old_vals = sub["jumlah_peserta_aktif"].tolist()
    new_vals = old_vals.copy()
    reduced_base = round(old_vals[break_idx] * (1 - drop_pct))
    actual_drop_amount = old_vals[break_idx] - reduced_base
    for i in range(break_idx, len(new_vals)):
        new_vals[i] = max(1, round(reduced_base * (1 + np.random.normal(0, 0.02))))
    headcount_df.loc[idxs, "jumlah_peserta_aktif"] = new_vals

    r_mask = (resign_df["employer_id"] == emp_id) & (resign_df["periode"] == sub["periode"].iloc[break_idx])
    resign_df.loc[r_mask, "jumlah_keluar"] = max(1, actual_drop_amount)

    mark(emp_id, "CLEAN_MUTASI_SAH",
         f"Headcount drop ~{drop_pct:.0%} di periode {sub['periode'].iloc[break_idx]}, DENGAN resign record matching "
         f"({actual_drop_amount} orang) -> true-negative test untuk confounder-handling Module A")

print("PDUK legit (true-negative) diinjeksi ke", len(pduk_legit_ids), "employer")

PDUK legit (true-negative) diinjeksi ke 10 employer


In [11]:
# --- 7c. Under-reporting wage: DPI ditekan ke 60-72% median cohort (peer, exclude diri sendiri) ---
for emp_id in wage_underreport_ids:
    cohort_row = employer_master.loc[employer_master["employer_id"] == emp_id].iloc[0]
    cohort_mask = ((employer_master["sektor_usaha"] == cohort_row["sektor_usaha"]) &
                   (employer_master["wilayah"] == cohort_row["wilayah"]) &
                   (employer_master["skala"] == cohort_row["skala"]))
    all_cohort_ids = employer_master.loc[cohort_mask, "employer_id"].tolist()
    peer_ids = [i for i in all_cohort_ids if i != emp_id] or all_cohort_ids

    # Median (bukan mean) dari rata-rata per-employer, exclude diri sendiri --
    # robust terhadap peer yang datanya lebih pendek (cold-start) atau peer lain
    # yang kebetulan juga sedang diinjeksi (contaminated baseline).
    per_employer_median = payroll_df[payroll_df["employer_id"].isin(peer_ids)].groupby("employer_id")["rata2_DPI"].median()
    cohort_median = per_employer_median.median()

    suppression = random.uniform(0.60, 0.72)
    target_wage = cohort_median * suppression
    mask = payroll_df["employer_id"] == emp_id
    n = mask.sum()
    payroll_df.loc[mask, "rata2_DPI"] = [round(target_wage * (1 + np.random.normal(0, 0.02)), -2) for _ in range(n)]
    mark(emp_id, "UNDER_REPORTING_WAGE",
         f"DPI ditekan ke ~{suppression:.0%} dari median cohort ({cohort_row['sektor_usaha']}/{cohort_row['wilayah']}/{cohort_row['skala']}, n_peer={len(peer_ids)})")

remittance_df = gen_remittance(payroll_df, headcount_df)  # re-generate karena payroll berubah
print("Under-reporting wage diinjeksi ke", len(wage_underreport_ids), "employer")

Under-reporting wage diinjeksi ke 25 employer


In [12]:
# --- 7d. Remittance gap: actual < expected, berulang >= 2 periode ---
for emp_id in remit_gap_ids:
    mask = remittance_df["employer_id"] == emp_id
    sub = remittance_df.loc[mask].sort_values("periode")
    idxs = sub.index.tolist()
    start_idx = random.randint(4, len(idxs) - 3)
    gap_pct = random.uniform(0.15, 0.30)
    new_actual = sub["actual_remittance"].tolist()
    for i in range(start_idx, len(new_actual)):
        new_actual[i] = round(sub["expected_contribution"].iloc[i] * (1 - gap_pct), -3)
    remittance_df.loc[idxs, "actual_remittance"] = new_actual
    mark(emp_id, "REMITTANCE_GAP",
         f"Actual remittance ~{gap_pct:.0%} di bawah expected, berulang sejak periode {sub['periode'].iloc[start_idx]}")

print("Remittance gap diinjeksi ke", len(remit_gap_ids), "employer")

Remittance gap diinjeksi ke 25 employer


## 8. `ground_truth.csv`

Tabel referensi employer_id → label anomali yang sengaja disisipkan. Dipakai untuk **validasi kuantitatif Wave 1** (precision/recall per modul + false-positive rate di counter-example) — bukan input untuk modul deteksi itu sendiri.

In [13]:
ground_truth_df = employer_master[["employer_id", "sektor_usaha", "wilayah", "skala"]].copy()
ground_truth_df["anomaly_type"] = ground_truth_df["employer_id"].map(lambda e: ground_truth[e]["anomaly_type"])
ground_truth_df["detail"] = ground_truth_df["employer_id"].map(lambda e: ground_truth[e]["detail"])

ground_truth_df["anomaly_type"].value_counts()

anomaly_type
CLEAN                   920
REMITTANCE_GAP           25
PDUK                     25
UNDER_REPORTING_WAGE     25
COLD_START               15
CLEAN_MUTASI_SAH         10
Name: count, dtype: int64

## 9. Sanity checks

Yang wajib dicek sebelum dataset ini dianggap siap dipakai di Wave 1:
1. Ukuran cohort (harus jauh di atas ~30, bukan cuma 2-3 seperti versi awal).
2. Tidak ada kolom yang menyerempet PII individu pekerja (constraint FR-6).
3. Kasus PDUK fraud vs PDUK legit benar-benar berbeda pola resign record-nya (kalau tidak beda, confounder-handling Module A tidak bisa diuji dengan bermakna).
4. Under-reporting wage benar-benar menghasilkan rasio upah yang jauh di bawah peer median.

In [14]:
print("=== Distribusi ground truth ===")
print(ground_truth_df["anomaly_type"].value_counts())

print("\n=== Ukuran cohort (min/median/max) ===")
cohort_sizes = employer_master.groupby(["sektor_usaha", "wilayah", "skala"]).size()
print("min:", cohort_sizes.min(), "median:", cohort_sizes.median(), "max:", cohort_sizes.max(),
      "n_cohort:", len(cohort_sizes))

print("\n=== Cek PII individu pekerja ===")
all_cols = (set(employer_master.columns) | set(headcount_df.columns) | set(payroll_df.columns)
            | set(remittance_df.columns) | set(resign_df.columns))
pii_like = [c for c in all_cols if any(k in c.lower() for k in ["nama", "nik", "ktp", "individu", "pekerja_id"])]
print("Kolom mencurigakan PII individu:", pii_like if pii_like else "TIDAK ADA (aman)")

=== Distribusi ground truth ===
anomaly_type
CLEAN                   920
REMITTANCE_GAP           25
PDUK                     25
UNDER_REPORTING_WAGE     25
COLD_START               15
CLEAN_MUTASI_SAH         10
Name: count, dtype: int64

=== Ukuran cohort (min/median/max) ===
min: 34 median: 34.0 max: 34 n_cohort: 30

=== Cek PII individu pekerja ===
Kolom mencurigakan PII individu: TIDAK ADA (aman)


In [15]:
print("--- Contoh PDUK fraud (resign harus rendah/0-1 di periode drop) ---")
e = pduk_fraud_ids[0]
hc = headcount_df[headcount_df.employer_id == e].sort_values("periode")
rs = resign_df[resign_df.employer_id == e].sort_values("periode")
display_df = pd.DataFrame({"periode": hc.periode.values, "headcount": hc.jumlah_peserta_aktif.values,
                            "resign": rs.jumlah_keluar.values})
print(display_df.to_string(index=False))

print("\n--- Contoh PDUK legit (resign harus ~cocok dengan magnitude drop) ---")
e = pduk_legit_ids[0]
hc = headcount_df[headcount_df.employer_id == e].sort_values("periode")
rs = resign_df[resign_df.employer_id == e].sort_values("periode")
display_df = pd.DataFrame({"periode": hc.periode.values, "headcount": hc.jumlah_peserta_aktif.values,
                            "resign": rs.jumlah_keluar.values})
print(display_df.to_string(index=False))

print("\n--- Contoh under-reporting wage (rasio upah sendiri vs peer median) ---")
e = wage_underreport_ids[0]
row = ground_truth_df[ground_truth_df.employer_id == e].iloc[0]
cohort = ground_truth_df[(ground_truth_df.sektor_usaha == row.sektor_usaha) &
                          (ground_truth_df.wilayah == row.wilayah) &
                          (ground_truth_df.skala == row.skala)].employer_id.tolist()
peers = [c for c in cohort if c != e]
peer_median = payroll_df[payroll_df.employer_id.isin(peers)].groupby("employer_id")["rata2_DPI"].median().median()
own = payroll_df[payroll_df.employer_id == e]["rata2_DPI"].mean()
print(f"{e}: own={own:,.0f}  peer_median(n={len(peers)})={peer_median:,.0f}  ratio={own/peer_median:.2f}")

--- Contoh PDUK fraud (resign harus rendah/0-1 di periode drop) ---
periode  headcount  resign
2025-01         35       0
2025-02         34       0
2025-03         34       1
2025-04         34       0
2025-05         34       1
2025-06         18       1
2025-07         18       0
2025-08         18       0
2025-09         18       0
2025-10         18       0
2025-11         18       0
2025-12         18       1

--- Contoh PDUK legit (resign harus ~cocok dengan magnitude drop) ---
periode  headcount  resign
2025-01         54       0
2025-02         55       1
2025-03         54       0
2025-04         53       1
2025-05         52       0
2025-06         35      18
2025-07         34       0
2025-08         34       0
2025-09         34       1
2025-10         35       0
2025-11         36       1
2025-12         36       0

--- Contoh under-reporting wage (rasio upah sendiri vs peer median) ---


EMP-0749: own=5,166,583  peer_median(n=33)=8,187,550  ratio=0.63


## 10. Simpan dataset

Enam file disimpan ke folder `ecrs_wave0_data/` — ini yang jadi input Wave 1 (Module A/B/C). Perhatikan `resign_records.csv` yang baru ditambahkan di v2.

In [16]:
employer_master.to_csv(f"{OUT_DIR}/employer_master.csv", index=False)
headcount_df.to_csv(f"{OUT_DIR}/headcount_timeseries.csv", index=False)
payroll_df.to_csv(f"{OUT_DIR}/payroll_timeseries.csv", index=False)
remittance_df.to_csv(f"{OUT_DIR}/remittance_timeseries.csv", index=False)
resign_df.to_csv(f"{OUT_DIR}/resign_records.csv", index=False)
ground_truth_df.to_csv(f"{OUT_DIR}/ground_truth.csv", index=False)

print("Tersimpan di folder:", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print(" -", f)

Tersimpan di folder: ecrs_wave0_data
 - employer_master.csv
 - ground_truth.csv
 - headcount_timeseries.csv
 - payroll_timeseries.csv
 - remittance_timeseries.csv
 - resign_records.csv


## 11. (Khusus Google Colab) Download hasil sebagai .zip

Sel ini otomatis di-skip kalau dijalankan di luar Colab (mis. Jupyter lokal) — tidak akan error.

In [17]:
import shutil

zip_path = shutil.make_archive("ecrs_wave0_data", "zip", OUT_DIR)
print("Zip dibuat:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Bukan environment Google Colab -- lewati auto-download.")
    print("File CSV tetap ada di folder:", OUT_DIR)

Zip dibuat: /mnt/attach/outputs/ecrs_wave0_data.zip
Bukan environment Google Colab -- lewati auto-download.
File CSV tetap ada di folder: ecrs_wave0_data


---

## Selanjutnya: Wave 1

Enam file di `ecrs_wave0_data/` (`employer_master.csv`, `headcount_timeseries.csv`, `payroll_timeseries.csv`, `remittance_timeseries.csv`, `resign_records.csv`, `ground_truth.csv`) adalah input untuk notebook Wave 1:

- **Module A** (Registration Volatility) — sekarang bisa memakai `resign_records.csv` untuk confounder-handling yang sesungguhnya, divalidasi terhadap 25 kasus PDUK fraud + 10 kasus PDUK legit (harus tidak diflag) + 15 kasus cold-start.
- **Module B** (Peer-Group Benchmarking) — pakai median/MAD (bukan mean/std) terhadap cohort ~34 anggota, divalidasi terhadap 25 kasus under-reporting wage.
- **Module C** (Contribution Reconciliation) — divalidasi terhadap 25 kasus remittance gap.
- Target validasi: precision & recall per modul + false-positive rate pada ~920 employer bersih (target <10%, sesuai target yang direvisi bersama).